# Tax revenue datasets — reproducible build

This notebook regenerates **both** tax-revenue CSVs in the data directory, pulling
everything live from source APIs so the files can be rebuilt at any time:

1. **`tax_revenue_2024_worldbank.csv`** — World Bank only, 2024 snapshot. Tax revenue
   (% of GDP), GDP and tax revenue in **current 2024 US dollars**, one row per country.
2. **`tax_revenue_harmonised_2018_2024.csv`** — hierarchical multi-source series
   (OECD Revenue Statistics → UNU-WIDER GRD 2025 → World Bank WDI). Most recent year
   in 2018–2024 per country, tax revenue **excluding social contributions**, % of GDP.

Requires internet access. Run top-to-bottom. Tested with pandas ≥ 2.0, requests, openpyxl.

> Re-running picks up the latest source vintages, so values can differ slightly from a
> prior snapshot as the OECD/GRD/World Bank publish new years.

In [1]:
import io
import time
from pathlib import Path

import pandas as pd
import requests

# Save outputs into the data directory. Defaults to the notebook's working directory;
# override DATA_DIR if you run this from elsewhere.
DATA_DIR = Path.cwd()
print("Writing outputs to:", DATA_DIR)

# ISO harmonisation: legacy Romania code -> current ISO.
def fix_iso(code):
    code = str(code).strip()
    return "ROU" if code == "ROM" else code

def get_json(url, tries=3):
    for i in range(tries):
        r = requests.get(url, timeout=120)
        if r.ok:
            return r.json()
        time.sleep(2)
    r.raise_for_status()

Writing outputs to: /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data


## Part 1 — World Bank 2024 dataset (current 2024 US dollars)

Single source (World Bank). Country universe = the 217 non-aggregate economies in the
World Bank country list. We pull:

- `GC.TAX.TOTL.GD.ZS` — tax revenue (% of GDP), 2024
- `NY.GDP.MKTP.CD` — GDP in current US dollars, 2024

and compute tax revenue in current 2024 USD = `pct/100 * GDP`.

In [2]:
# 1.1  Country universe: World Bank economies excluding regional/income aggregates.
WB = "https://api.worldbank.org/v2/"
meta = get_json(WB + "country?format=json&per_page=400")[1]
countries = [c for c in meta if c["region"]["value"] != "Aggregates"]
wb_name = {c["id"]: c["name"] for c in countries}          # iso3 -> name
AGG_CODES = {c["id"] for c in meta if c["region"]["value"] == "Aggregates"}
# OECD SDMX also carries non-country aggregate codes absent from the WB list.
AGG_CODES |= {"A9", "F", "OECD_REP", "SO"}
print(f"{len(countries)} non-aggregate World Bank economies")

217 non-aggregate World Bank economies


In [3]:
# 1.2  Tax revenue (% of GDP), 2024.
tax_rows = get_json(
    WB + "country/all/indicator/GC.TAX.TOTL.GD.ZS?date=2024&format=json&per_page=400"
)[1]
tax_2024 = {r["countryiso3code"]: r["value"] for r in tax_rows if r["countryiso3code"]}

# 1.3  GDP in current US dollars, 2024.
gdp_rows = get_json(
    WB + "country/all/indicator/NY.GDP.MKTP.CD?date=2024&format=json&per_page=400"
)[1]
gdp_2024 = {r["countryiso3code"]: r["value"] for r in gdp_rows if r["countryiso3code"]}

print("tax obs:", sum(v is not None for v in tax_2024.values()),
      " gdp obs:", sum(v is not None for v in gdp_2024.values()))

tax obs: 96  gdp obs: 236


In [4]:
# 1.4  Assemble, compute USD tax, save. One row per economy, sorted by ISO3.
rows = []
for iso in sorted(wb_name):
    pct = tax_2024.get(iso)
    gdp = gdp_2024.get(iso)
    tax_usd = (pct / 100.0 * gdp) if (pct is not None and gdp is not None) else None
    rows.append({
        "country_name": wb_name[iso],
        "iso3_code": iso,
        "tax_revenue_pct_gdp": pct,
        "gdp_current_usd_2024": gdp,
        "tax_revenue_current_usd_2024": tax_usd,
        "unit": "current 2024 US dollars",
    })

wb_df = pd.DataFrame(rows)
wb_path = DATA_DIR / "tax_revenue_2024_worldbank.csv"
wb_df.to_csv(wb_path, index=False)

print("Saved", wb_path.name, "->", len(wb_df), "countries")
print("  with tax %% GDP:", wb_df["tax_revenue_pct_gdp"].notna().sum())
print("  with GDP:", wb_df["gdp_current_usd_2024"].notna().sum())
print("  with tax in USD:", wb_df["tax_revenue_current_usd_2024"].notna().sum())
wb_df.head()

Saved tax_revenue_2024_worldbank.csv -> 217 countries
  with tax %% GDP: 77
  with GDP: 192
  with tax in USD: 76


,country_name,iso3_code,tax_revenue_pct_gdp,gdp_current_usd_2024,tax_revenue_current_usd_2024,unit
0,Aruba,ABW,NaN,4.265651e+09,NaN,current 2024 US dollars
1,Afghanistan,AFG,NaN,NaN,NaN,current 2024 US dollars
2,Angola,AGO,7.744364,1.009989e+11,7.821723e+09,current 2024 US dollars
3,Albania,ALB,18.015293,2.704643e+10,4.872494e+09,current 2024 US dollars
4,Andorra,AND,15.586312,4.039842e+09,6.296624e+08,current 2024 US dollars


## Part 2 — Harmonised 2018–2024 dataset

Tax revenue **excluding social contributions**, % of GDP. For each country we take the
first available source in priority order, and within that source the most recent year in
2018–2024:

1. **OECD Revenue Statistics** (SDMX `DSD_REV_COMP_GLOBAL`): general government (`S13`),
   % of GDP (`PT_B1GQ`); total tax (`_T`) minus social contributions (`T_2000`).
2. **UNU-WIDER GRD 2025** (`PCTGDP` sheet): the *Taxes — Excluding SC* column × 100.
3. **World Bank WDI** (`GC.TAX.TOTL.GD.ZS`): most recent non-null year.

Output columns: `iso3_code, country_name, tax_revenue_pct_gdp, data_year, source, definition`.

In [5]:
# 2.1  OECD Revenue Statistics via SDMX (comparable global dataflow).
# ~85 MB CSV; this cell can take a minute.
OECD_URL = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.CTP.TPS,DSD_REV_COMP_GLOBAL@DF_RSGLOBAL,/all"
    "?startPeriod=2018&endPeriod=2024&dimensionAtObservation=AllDimensions"
)
resp = requests.get(
    OECD_URL,
    headers={"Accept": "application/vnd.sdmx.data+csv; charset=utf-8"},
    timeout=600,
)
resp.raise_for_status()
oecd_raw = pd.read_csv(io.StringIO(resp.text), low_memory=False)
print("OECD rows:", len(oecd_raw))

# General government, % of GDP.
o = oecd_raw[(oecd_raw.SECTOR == "S13") & (oecd_raw.UNIT_MEASURE == "PT_B1GQ")]
o = o[o.TIME_PERIOD.between(2018, 2024)]

total = (o[o.STANDARD_REVENUE == "_T"][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]
         .rename(columns={"OBS_VALUE": "total_tax"}))
ssc = (o[o.STANDARD_REVENUE == "T_2000"][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]
       .rename(columns={"OBS_VALUE": "ssc"}))

oecd = total.merge(ssc, on=["REF_AREA", "TIME_PERIOD"], how="left")
oecd["ssc"] = oecd["ssc"].fillna(0)                       # tax excl. social contributions
oecd["value"] = oecd["total_tax"] - oecd["ssc"]
oecd = oecd.dropna(subset=["value"])
oecd["iso3_code"] = oecd["REF_AREA"].map(fix_iso)
# Most recent year per country.
oecd = (oecd.sort_values("TIME_PERIOD").groupby("iso3_code", as_index=False).tail(1)
            .rename(columns={"TIME_PERIOD": "data_year"}))
oecd = oecd[["iso3_code", "value", "data_year"]]
oecd["source"] = "OECD Revenue Statistics"
oecd["definition"] = ("Tax revenue excl. SSC, %% GDP "
                      "(general government, OECD Revenue Statistics)").replace("%%", "%")
print("OECD countries:", len(oecd))

OECD rows: 762711
OECD countries: 144


In [6]:
# 2.2  UNU-WIDER Government Revenue Dataset 2025 (general government, % of GDP).
GRD_URL = "https://www.wider.unu.edu/sites/default/files/Data/UNUWIDERGRD_2025_General.xlsx"
hdr = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Referer": "https://www.wider.unu.edu/database/data-and-resources-grd",
}
r = requests.get(GRD_URL, headers=hdr, timeout=300)
r.raise_for_status()
grd_raw = pd.read_excel(io.BytesIO(r.content), sheet_name="PCTGDP")

# On the PCTGDP sheet, column index 25 is "Taxes / Excluding SC" (a sub-column under
# the merged "Taxes" header). Values are fractions of GDP -> multiply by 100.
grd = pd.DataFrame({
    "iso3_code": grd_raw["iso"].map(fix_iso),
    "country": grd_raw["country"],
    "data_year": pd.to_numeric(grd_raw["year"], errors="coerce"),
    "value": pd.to_numeric(grd_raw.iloc[:, 25], errors="coerce") * 100,
}).dropna(subset=["iso3_code", "data_year", "value"])
grd["data_year"] = grd["data_year"].astype(int)
grd = grd[grd.data_year.between(2018, 2024)]
grd_name = dict(zip(grd["iso3_code"], grd["country"]))     # iso3 -> GRD name
grd = (grd.sort_values("data_year").groupby("iso3_code", as_index=False).tail(1))
grd = grd[["iso3_code", "value", "data_year"]]
grd["source"] = "UNU-WIDER GRD 2025"
grd["definition"] = "Tax revenue excl. SSC, % GDP (general government, UNU-WIDER GRD 2025)"
print("GRD countries:", len(grd))

GRD countries: 148


In [7]:
# 2.3  World Bank WDI fallback: most recent non-null year in 2018-2024.
wdi_rows = get_json(
    WB + "country/all/indicator/GC.TAX.TOTL.GD.ZS?date=2018:2024&format=json&per_page=20000"
)[1]
wdi = pd.DataFrame([
    {"iso3_code": fix_iso(r["countryiso3code"]),
     "data_year": int(r["date"]),
     "value": r["value"]}
    for r in wdi_rows
    if r["value"] is not None and r["countryiso3code"]
])
wdi = wdi[wdi.data_year.between(2018, 2024)]
wdi = (wdi.sort_values("data_year").groupby("iso3_code", as_index=False).tail(1))
wdi["source"] = "World Bank WDI"
wdi["definition"] = ("Tax revenue excl. SSC, % GDP "
                     "(central government, World Bank WDI / IMF GFS)")
print("WDI countries:", len(wdi))

WDI countries: 174


In [8]:
# 2.4  Hierarchical merge: OECD > GRD > WDI. First available source wins per country.
priority = [oecd, grd, wdi]
picked = {}
for src in priority:
    for _, row in src.iterrows():
        iso = row["iso3_code"]
        if iso and iso not in AGG_CODES and iso not in picked:
            picked[iso] = row

def name_for(iso):
    # World Bank name preferred; GRD name fills Kosovo etc.; blank if neither has it.
    return wb_name.get(iso) or grd_name.get(iso) or ""

harm = pd.DataFrame([
    {
        "iso3_code": iso,
        "country_name": name_for(iso),
        "tax_revenue_pct_gdp": round(float(r["value"]), 4),
        "data_year": int(r["data_year"]),
        "source": r["source"],
        "definition": r["definition"],
    }
    for iso, r in picked.items()
]).sort_values("iso3_code").reset_index(drop=True)

harm_path = DATA_DIR / "tax_revenue_harmonised_2018_2024.csv"
harm.to_csv(harm_path, index=False, encoding="utf-8")
print("Saved", harm_path.name, "->", len(harm), "countries (no duplicate ISOs:",
      harm["iso3_code"].is_unique, ")")
harm.head()

Saved tax_revenue_harmonised_2018_2024.csv -> 175 countries (no duplicate ISOs: True )


,iso3_code,country_name,tax_revenue_pct_gdp,data_year,source,definition
0,AGO,Angola,7.7444,2024,World Bank WDI,"Tax revenue excl. SSC, % GDP (central governme..."
1,ALB,Albania,19.3350,2023,UNU-WIDER GRD 2025,"Tax revenue excl. SSC, % GDP (general governme..."
2,AND,Andorra,15.5863,2024,World Bank WDI,"Tax revenue excl. SSC, % GDP (central governme..."
3,ARE,United Arab Emirates,18.0037,2022,UNU-WIDER GRD 2025,"Tax revenue excl. SSC, % GDP (general governme..."
4,ARG,Argentina,22.4874,2024,OECD Revenue Statistics,"Tax revenue excl. SSC, % GDP (general governme..."


### Validation report

In [9]:
# Coverage, year distribution, source distribution, and EXT-host check.
EXT_HOSTS = ["ARG","AUS","BRA","CHL","CHN","CMR","COD","COL","CRI","CUB","ETH","GTM",
             "HND","HTI","IDN","IND","JPN","LKA","MDG","MEX","MUS","MYS","NZL","PAN",
             "PER","PHL","PNG","TZA","USA","VEN","VNM","ZAF"]

print("(1) Total countries with tax data:", len(harm))

covered = harm[harm.iso3_code.isin(EXT_HOSTS)]
missing = sorted(set(EXT_HOSTS) - set(covered.iso3_code))
print(f"(2) EXT >= 0.1 hosts covered: {len(covered)} of {len(EXT_HOSTS)}")
print("    missing:", missing or "none")

print("(3) data_year distribution:")
print(harm["data_year"].value_counts().sort_index().to_string())

print("(4) source distribution:")
print(harm["source"].value_counts().to_string())

print("(5) EXT hosts on fallback sources (non-OECD):")
fb = covered[covered.source != "OECD Revenue Statistics"]
print(fb[["iso3_code", "source", "data_year", "tax_revenue_pct_gdp"]].to_string(index=False)
      if len(fb) else "    none")

(1) Total countries with tax data: 175
(2) EXT >= 0.1 hosts covered: 30 of 32
    missing: ['HTI', 'VEN']
(3) data_year distribution:
data_year
2018     2
2019     2
2020     4
2021     3
2022     4
2023    61
2024    99
(4) source distribution:
source
OECD Revenue Statistics    140
UNU-WIDER GRD 2025          22
World Bank WDI              13
(5) EXT hosts on fallback sources (non-OECD):
iso3_code             source  data_year  tax_revenue_pct_gdp
      ETH UNU-WIDER GRD 2025       2023               6.8231
      IND UNU-WIDER GRD 2025       2020              16.1599
      TZA     World Bank WDI       2024              12.8029
